# Day 22: Embeddings, Vector DBs and the RAG App
**Embeddings, flat, HNSW and IVF indexes, hybrid search, re-ranking, and choosing RAG, fine-tuning, long context or caching**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas faiss-cpu rank_bm25

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas faiss-cpu rank_bm25
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. Documents and chunks

In [ ]:
%%writefile knowledge.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

## IT FAQ
Error VPN-417 means your VPN certificate has expired. Renew it from the IT self-service portal and restart the VPN client.
Password resets are done through the self-service portal. Passwords must be at least 14 characters long.
Approved AI tools are listed on the IT intranet page. Do not paste customer data into tools that are not on that list.
New laptops are issued within 5 working days of a request approved by your manager.

## Travel FAQ
Book flights through the corporate travel desk at least 10 days before travel. Economy class is standard for flights under 6 hours.
Hotel stays are reimbursed up to 6,000 INR per night in India and 15,000 JPY per night in Japan.
Visa support letters are issued by HR within 3 working days.

In [ ]:
import re, time, numpy as np, pandas as pd
text = open("knowledge.md").read()
chunks = []
for sec in re.split(r"\n(?=## )", text):
    lines = sec.strip().splitlines()
    if not lines or not lines[0].startswith("## "):
        continue
    title = lines[0][3:].strip()
    for s in re.split(r"(?<=[.!?])\s+", " ".join(lines[1:]).strip()):
        if s:
            chunks.append({"id": f"c{len(chunks) + 1}", "section": title, "text": s})
texts = [f"{c['section']}: {c['text']}" for c in chunks]
print(len(chunks), "chunks. Example:", chunks[0])

## 3. Embeddings, cosine similarity and distance

In [ ]:
sents = ["How many leave days do I get?", "Employees receive 18 days of paid leave per year.",
         "What is the hotel limit in Tokyo?", "Hotel stays are reimbursed up to 15,000 JPY per night in Japan.",
         "How long must my password be?", "Passwords must be at least 14 characters long."]
V = embed(sents)
sim = pd.DataFrame(V @ V.T, index=[s[:22] for s in sents], columns=[f"s{i}" for i in range(6)]).round(2)
print("Cosine similarity\n", sim)
print("\nCosine distance (1 - similarity), first row:", (1 - sim.iloc[0]).round(2).tolist())
if not ONLINE:
    print("\nOffline embeddings only match shared words. With a real model, questions match their answers by meaning.")

## 4. A vector index with FAISS

In [ ]:
import faiss
vecs = embed(texts).astype("float32")
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)

def vector_search(q, k=5):
    D, I = index.search(embed(q).astype("float32"), k)
    return [chunks[i]["id"] for i in I[0]]

for q in ["hotel limit in Japan", "VPN-417 error", "carry forward leave"]:
    print(f"{q:<22} -> {vector_search(q, 3)}")

## 5. Exact vs approximate search at scale: flat, HNSW and IVF

In [ ]:
rng = np.random.default_rng(23)
X = rng.normal(size=(20000, 128)).astype("float32"); X /= np.linalg.norm(X, axis=1, keepdims=True)
Q = rng.normal(size=(200, 128)).astype("float32"); Q /= np.linalg.norm(Q, axis=1, keepdims=True)

def timed(idx):
    t = time.perf_counter(); _, I = idx.search(Q, 10)
    return I, (time.perf_counter() - t) / len(Q) * 1000

flat = faiss.IndexFlatIP(128); flat.add(X)
I_flat, t_flat = timed(flat)
recall = lambda I: np.mean([len(set(a) & set(b)) / 10 for a, b in zip(I_flat, I)])
rows = [{"index": "Flat (exact)", "ms/query": t_flat, "recall@10": 1.0}]

hnsw = faiss.IndexHNSWFlat(128, 32, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 80
hnsw.add(X)
for ef in [16, 64]:                        # efSearch: how widely HNSW explores per query
    hnsw.hnsw.efSearch = ef
    I, t = timed(hnsw); rows.append({"index": f"HNSW efSearch={ef}", "ms/query": t, "recall@10": recall(I)})

quant = faiss.IndexFlatIP(128)
ivf = faiss.IndexIVFFlat(quant, 128, 128, faiss.METRIC_INNER_PRODUCT)   # 128 clusters
ivf.train(X); ivf.add(X)
for nprobe in [4, 16]:                     # nprobe: how many clusters to search
    ivf.nprobe = nprobe
    I, t = timed(ivf); rows.append({"index": f"IVF nprobe={nprobe}", "ms/query": t, "recall@10": recall(I)})

print(pd.DataFrame(rows).round(3).to_string(index=False))
print("Random vectors are a hard case. Real embeddings cluster by topic, so ANN recall is usually higher at the same speed.")

## 6. Metadata filters
Filter by metadata **before** ranking. Permissions work the same way (Day 18).

In [ ]:
def filtered_search(q, k=3, section=None):
    pool = [i for i, c in enumerate(chunks) if section is None or c["section"] == section]
    sims = vecs[pool] @ embed(q)[0].astype("float32")
    return [chunks[pool[i]]["id"] for i in np.argsort(sims)[::-1][:k]]

print("All sections :", filtered_search("limit per night"))
print("Travel FAQ   :", filtered_search("limit per night", section="Travel FAQ"))

## 7. Keyword search with BM25

In [ ]:
from rank_bm25 import BM25Okapi
tok = lambda s: re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", s.lower())
bm25 = BM25Okapi([tok(t) for t in texts])

def bm25_search(q, k=5):
    scores = bm25.get_scores(tok(q))
    return [chunks[i]["id"] for i in np.argsort(scores)[::-1][:k]]

for q in ["VPN-417", "14 characters", "visa letter"]:
    print(f"{q:<14} bm25 {bm25_search(q, 3)}   vector {vector_search(q, 3)}")

## 8. Hybrid search with reciprocal rank fusion, measured on ten questions

In [ ]:
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, 1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_search(q, k=5):
    return rrf([bm25_search(q, 20), vector_search(q, 20)])[:k]

find = lambda frag: next(c["id"] for c in chunks if frag in c["text"])
tests = [("How many paid leave days do full-time staff get?", find("18 days of paid leave")),
         ("When does carried over leave expire?", find("expires on 31 March")),
         ("What does error VPN-417 mean?", find("VPN-417")),
         ("How long must a password be?", find("14 characters")),
         ("Hotel limit per night in Japan?", find("15,000 JPY")),
         ("How soon must I book flights?", find("10 days before travel")),
         ("Deadline to submit travel expenses?", find("within 30 days")),
         ("Who issues visa support letters?", find("Visa support letters")),
         ("How many sick days do employees get?", find("8 days of paid sick leave")),
         ("By when must remote work be logged?", find("logged in the HR portal"))]

def hit_rate(search, k=3):
    return np.mean([gold in search(q, k) for q, gold in tests])

def mrr(search, k=10):
    out = []
    for q, gold in tests:
        res = search(q, k)
        out.append(1 / (res.index(gold) + 1) if gold in res else 0)
    return np.mean(out)

table = pd.DataFrame({m: {"hit@3": hit_rate(f), "MRR": mrr(f)} for m, f in
                      {"bm25": bm25_search, "vector": vector_search, "hybrid": hybrid_search}.items()}).T.round(2)
print(table)

## 9. Re-ranking

In [ ]:
by_id = {c["id"]: c for c in chunks}

def rerank(q, ids, keep=3):
    scored = []
    for cid in ids:
        c = by_id[cid]
        if ONLINE:
            r = llm(f"Rate 0 to 10 how well this passage answers the question. Reply with one integer.\n"
                    f"Question: {q}\nPassage: {c['text']}", temperature=0)
            nums = re.findall(r"\d+", r)
            s = float(nums[0]) / 10 if nums else 0.0
        else:
            qw, cw = set(tok(q)), set(tok(c["section"] + " " + c["text"]))
            s = len(qw & cw) / max(1, len(qw))           # offline stand-in for a cross-encoder
        scored.append((cid, round(s, 2)))
    return sorted(scored, key=lambda x: x[1], reverse=True)[:keep]

reranked = lambda q, k=3: [cid for cid, _ in rerank(q, hybrid_search(q, 8), keep=k)]
table.loc["hybrid + re-rank"] = [hit_rate(reranked), mrr(reranked)]
print(table.round(2))
q = "When does carried over leave expire?"
print("\nCandidates:", hybrid_search(q, 8))
print("Re-ranked: ", rerank(q, hybrid_search(q, 8)))

## 10. Grounded answers with a refusal threshold

In [ ]:
RULES = "Answer using ONLY the context. Cite chunk ids like [c3]. If the context does not contain the answer, say \"I don't know\"."

def ask(q, threshold=0.3):
    top = rerank(q, hybrid_search(q, 8))
    if not top or top[0][1] < threshold:
        return "I don't know (no relevant passage found).", top
    context = "\n".join(f"[{cid}] {by_id[cid]['text']}" for cid, _ in top)
    return llm(f"Context:\n{context}\n\nQuestion: {q}", system=RULES, temperature=0), top

for q in ["What does error VPN-417 mean?", "Hotel limit per night in Japan?", "Can I bring my dog to the office?"]:
    a, top = ask(q)
    print(f"Q: {q}\nA: {a}\n   evidence: {top}\n")

### TODO 1: Add two more test questions (one unanswerable) and tune `threshold` so answerable questions are answered and the unanswerable one is refused.

In [ ]:
# TODO: write your code here

## 11. NEW (v4): RAG vs fine-tuning vs long context vs caching
A rule-based helper plus a monthly token cost comparison. Prices are **placeholders**, not real prices. Real caching rules (minimum prefix length, cache lifetime, discount) vary by provider and model; check the documentation.

In [ ]:
PRICE_PER_1M = {"input": 1.00, "cached_input": 0.10, "output": 4.00}   # PLACEHOLDER prices

def monthly_cost(corpus_tokens, queries_per_day, rag_context=1500, q_tokens=50, out_tokens=200, days=30):
    n = queries_per_day * days
    p = PRICE_PER_1M
    long_ctx = n * ((corpus_tokens + q_tokens) * p["input"] + out_tokens * p["output"]) / 1e6
    cached = n * (corpus_tokens * p["cached_input"] + q_tokens * p["input"] + out_tokens * p["output"]) / 1e6
    rag = n * ((rag_context + q_tokens) * p["input"] + out_tokens * p["output"]) / 1e6
    return {"long context": round(long_ctx, 2), "long context + cache": round(cached, 2), "RAG": round(rag, 2)}

def recommend(corpus_tokens, changes_per_month, queries_per_day, goal="facts", whole_doc=False, window=200_000):
    if goal == "style":
        return "fine-tuning or examples (style, not facts)"
    if corpus_tokens > window * 0.5:
        return "RAG (too big for the window)"
    if changes_per_month > 4:
        return "RAG (changes often: re-index, no retrain)"
    if whole_doc:
        return "long context + cache (needs the whole document)" if queries_per_day > 20 else "long context"
    cost = monthly_cost(corpus_tokens, queries_per_day)
    best = min(["long context + cache", "RAG"], key=cost.get)
    return f"{best} (cheapest)"

scenarios = [   # name, corpus tokens, changes per month, queries per day, goal, needs the whole document
    ("20-page policy, 50/day", 12_000, 1, 50, "facts", False),
    ("2,000-page manual, weekly updates", 1_200_000, 4, 3_000, "facts", False),
    ("brand tone for replies", 0, 0, 2_000, "style", False),
    ("90k token contract review, 5,000/day", 90_000, 0, 5_000, "facts", True),
    ("catalogue changing hourly", 300_000, 720, 8_000, "facts", False),
]
rows = []
for name, corpus, changes, qpd, goal, whole in scenarios:
    rec = recommend(corpus, changes, qpd, goal, whole)
    cost = monthly_cost(corpus, qpd) if goal == "facts" else {}
    rows.append({"scenario": name, "recommend": rec, **cost})
print(pd.DataFrame(rows).to_string(index=False))
print("\nCosts are in placeholder units per month. Long context over a huge corpus may not fit the window at all.")

### TODO 2 (SmartDesk step 19): run keyword, vector, hybrid and hybrid + re-rank on ten SmartDesk FAQ questions. Paste the hit-rate table for the hotel lab.

In [ ]:
# TODO: write your code here

## 12. Check yourself
1. Which method won on codes like VPN-417, and why?
2. Did re-ranking improve MRR?
3. For which scenario does caching beat RAG on cost, and what would make you still choose RAG?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)